# Early Dementia Detection from Brain MRI
### YOLOv8 Classification Benchmark
**Author:** Claude (Adaptive AI Collaborator)
**Methodology:**
* **Data:** OASIS MRI Dataset, same patient-level splits as `notebooks/Dementia.ipynb`.
* **Architecture:** YOLOv8n-cls, fine-tuned from COCO-pretrained classification weights.
* **Robustness:** Reuses `src/data_pipeline.py` so the CNN and YOLOv8 are benchmarked on identical patients.
* **Goal:** Reproduce the documented **91.64% accuracy / 0.7869 Scott's Pi / 0.8447 QWK**, and only replace `models/best.pt` after a side-by-side comparison is approved.


## 1. Setup
Install `ultralytics` (which bundles the YOLOv8 classification trainer) and import the libraries used throughout this notebook.


In [ ]:
%pip install -q ultralytics

import os
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from ultralytics import YOLO

print("Ultralytics + torch ready. CUDA available:", torch.cuda.is_available())


## 2. Load Metadata & Patient-Level Splits
Rather than re-implementing metadata extraction and splitting, we import the exact same functions used by the CNN pipeline from **`src/data_pipeline.py`**. This guarantees YOLOv8 is trained and evaluated on the *identical* patients as `notebooks/Dementia.ipynb` — same `random_state=42`, same 80/10/10-style patient-level split — so the two models' results are directly comparable.


In [ ]:
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(REPO_ROOT / "src"))

from data_pipeline import CATEGORIES, create_metadata_df, patient_level_split, hybrid_resample
from evaluate import evaluate_predictions

# Update this to wherever the OASIS `Data/<category>/...jpg` folders live on disk
DATA_PATH = REPO_ROOT / "dataset" / "Data"

df = create_metadata_df(DATA_PATH)
train_df, val_df, test_df = patient_level_split(df)

print(f"Total scans: {len(df)} | Unique patients: {df['patient_id'].nunique()}")
print(f"Training:   {train_df['patient_id'].nunique()} patients | {len(train_df)} scans")
print(f"Validation: {val_df['patient_id'].nunique()} patients | {len(val_df)} scans")
print(f"Testing:    {test_df['patient_id'].nunique()} patients | {len(test_df)} scans")


## 3. Hybrid Resampling (Training Split Only)
Same **137:1 imbalance** problem as the CNN, so we reuse `hybrid_resample` from `src/data_pipeline.py` with the same `target_samples_per_class=8000`. Validation and test stay at their natural distribution — resampling only the training split keeps the test-set accuracy an honest estimate of real-world performance.


In [ ]:
train_df_balanced = hybrid_resample(train_df, target_samples_per_class=8000)

print("Balanced training distribution:")
print(train_df_balanced["category"].value_counts())


## 4. Organize Images into YOLO's Classification Folder Structure
YOLOv8 classification mode expects **ImageFolder-style directories** (`<root>/train/<class>/*.jpg`, `<root>/val/<class>/*.jpg`, `<root>/test/<class>/*.jpg`), not a dataframe. `organize_into_yolo_folders` below materializes our three splits into that layout under `yolo_data/` (already listed in `.gitignore`, since it's a derived cache, not source data).

We use symlinks where the OS allows it (Linux/Colab) to avoid duplicating tens of thousands of images on disk, falling back to a real copy only where symlinks aren't permitted (e.g. Windows without Developer Mode). The function is idempotent — re-running it skips files that already exist.


In [ ]:
def organize_into_yolo_folders(splits, output_root):
    output_root = Path(output_root)
    for split_name, split_df in splits.items():
        for i, row in enumerate(tqdm(split_df.itertuples(), total=len(split_df), desc=f"Organizing {split_name}")):
            dest_dir = output_root / split_name / row.category
            dest_dir.mkdir(parents=True, exist_ok=True)
            dest_path = dest_dir / f"{i}_{os.path.basename(row.path)}"
            if dest_path.exists():
                continue
            try:
                os.symlink(os.path.abspath(row.path), dest_path)
            except OSError:
                shutil.copy2(row.path, dest_path)

YOLO_DATA_ROOT = REPO_ROOT / "yolo_data"
organize_into_yolo_folders(
    {"train": train_df_balanced, "val": val_df, "test": test_df},
    YOLO_DATA_ROOT,
)

print("YOLO folder structure ready at:", YOLO_DATA_ROOT)


## 5. Load a Pretrained YOLOv8 Classification Checkpoint
We start from `yolov8n-cls.pt` — pretrained on ImageNet — the same nano-sized checkpoint already documented in `app.py` (1.44M params). Ultralytics downloads it automatically on first use.


In [ ]:
model = YOLO("yolov8n-cls.pt")


## 6. Fine-Tune on OASIS
We mirror the CNN's training philosophy from `notebooks/Dementia.ipynb`:
* **`epochs=50`** — same epoch budget as `model.fit(..., epochs=50, ...)` for the CNN.
* **`patience=5`** — YOLOv8's built-in early stopping plays the same role as Keras's `EarlyStopping(patience=5, monitor='val_loss')`: training stops if fitness hasn't improved for 5 consecutive epochs, and the best epoch's weights are kept.
* **`imgsz=128`** — matches the CNN's `IMG_SIZE=(128, 128)` and the `imgsz=128` already used for YOLO inference in `app.py`.

Weights are written under `runs/classify/oasis_yolo_candidate/weights/`, **not** `models/best.pt` — we don't touch the incumbent checkpoint until the comparison in Section 9 is approved.


In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"

train_results = model.train(
    data=str(YOLO_DATA_ROOT),
    epochs=50,
    patience=5,
    imgsz=128,
    batch=32,
    seed=42,
    device=device,
    project=str(REPO_ROOT / "runs" / "classify"),
    name="oasis_yolo_candidate",
    exist_ok=True,
    pretrained=True,
)


## 7. Candidate Weights — Saved Separately, Not Overwriting `models/best.pt`
The best checkpoint from this run is copied to `models/yolo_candidate.pt` — a separate path from the incumbent `models/best.pt`. We only decide whether to promote it after evaluating both side by side (Section 9).


In [ ]:
candidate_source = Path(train_results.save_dir) / "weights" / "best.pt"
CANDIDATE_PATH = REPO_ROOT / "models" / "yolo_candidate.pt"

shutil.copy2(candidate_source, CANDIDATE_PATH)
print("Candidate weights saved to:", CANDIDATE_PATH)


## 8. Evaluate on the Test Split
First, Ultralytics' built-in `model.val()` gives Top-1 / Top-5 accuracy on the held-out test patients directly from the folder structure.


In [ ]:
candidate_model = YOLO(CANDIDATE_PATH)

val_metrics = candidate_model.val(data=str(YOLO_DATA_ROOT), split="test", imgsz=128)
print(f"Ultralytics Top-1 accuracy: {val_metrics.top1 * 100:.2f}%")
print(f"Ultralytics Top-5 accuracy: {val_metrics.top5 * 100:.2f}%")


## 9. Manual Scott's Pi & Quadratic Weighted Kappa
The documented benchmark (91.64% accuracy / 0.7869 Scott's Pi / 0.8447 QWK) uses the same manual metric functions as `src/evaluate.py`, not Ultralytics' built-in metrics. We run predictions ourselves on the raw test dataframe (not the copied folder) so we keep the ordinal `label` column, map YOLO's alphabetically-sorted class indices back to our `CATEGORIES` ordinal encoding, and then call `evaluate_predictions`.


In [ ]:
def predict_and_score(yolo_model, eval_df):
    y_true = eval_df["label"].to_numpy()
    y_pred = []
    for result in yolo_model.predict(eval_df["path"].tolist(), imgsz=128, batch=32, stream=True, verbose=False):
        class_name = yolo_model.names[int(result.probs.top1)]
        y_pred.append(CATEGORIES[class_name])
    return y_true, np.array(y_pred)

y_true, y_pred = predict_and_score(candidate_model, test_df)
candidate_metrics = evaluate_predictions(y_true, y_pred, num_classes=4)

DOCUMENTED_TARGETS = {"accuracy": 0.9164, "scotts_pi": 0.7869, "qwk": 0.8447}

print("Candidate vs documented target:")
for key, target in DOCUMENTED_TARGETS.items():
    value = candidate_metrics[key]
    print(f"  {key:10s}: {value:.4f}  (target {target:.4f}, delta {value - target:+.4f})")


## 10. Compare Candidate vs Incumbent `models/best.pt`
Before deciding whether to promote the new checkpoint, we run the **identical** evaluation procedure (Sections 8-9) against the current `models/best.pt`, so both numbers come from the same test patients and the same metric code — no other difference.


In [ ]:
def evaluate_checkpoint(weights_path):
    yolo_model = YOLO(weights_path)
    y_true, y_pred = predict_and_score(yolo_model, test_df)
    return evaluate_predictions(y_true, y_pred, num_classes=4)

INCUMBENT_PATH = REPO_ROOT / "models" / "best.pt"

comparison_rows = {"candidate (yolo_candidate.pt)": candidate_metrics}
if INCUMBENT_PATH.exists():
    comparison_rows["incumbent (best.pt)"] = evaluate_checkpoint(INCUMBENT_PATH)
else:
    print(f"No incumbent checkpoint found at {INCUMBENT_PATH} — nothing to compare against yet.")
comparison_rows["documented target"] = DOCUMENTED_TARGETS

comparison_df = pd.DataFrame(comparison_rows).T
comparison_df


## 11. Promote to `models/best.pt` — Manual Approval Required
**Do not run the cell below until you've reviewed the comparison table above.** `CONFIRM_OVERWRITE` defaults to `False`, so re-running the whole notebook never silently replaces the incumbent checkpoint. Flip it to `True` only after confirming the candidate reproduces or exceeds `models/best.pt`'s documented performance.


In [ ]:
CONFIRM_OVERWRITE = False  # Flip to True only after reviewing Section 10's comparison table

if CONFIRM_OVERWRITE:
    shutil.copy2(CANDIDATE_PATH, INCUMBENT_PATH)
    print(f"models/best.pt updated from {CANDIDATE_PATH}")
else:
    print(f"Skipped — models/best.pt left untouched. Candidate weights remain at {CANDIDATE_PATH}")
